# Install required libraries

In [ ]:
# !pip install klib lazypredict dask[dataframe] dill shap deap

# Save the state of the notebook

In [ ]:
# import dill
#
# dill.dump_session("data_analysis.pkl")

# Load the state of the notebook

In [ ]:
# import dill
#
# dill.load_session("data_analysis.pkl")

# read all the csv files in the data folder and concatenate them into one

In [ ]:
import os
import glob
import gc
import pandas as pd

path = "data/kdd99"
all_files = glob.glob(os.path.join(path, "*.csv"))

df_from_each_file = (pd.read_csv(f) for f in all_files)
df = pd.concat(df_from_each_file, ignore_index=True)
del df_from_each_file
gc.collect()

# Exploratory Data Analysis

## View basic information about the data

In [ ]:
from IPython.display import display
import numpy as np

display(df.info())
display(df.head())

## how many records does the data have?

In [ ]:
display("Number of records: ", df.shape[0])

## how many features does the data have?

In my project, I will consider all the features except the ' Label' as the
features.

In [ ]:
display("Number of features: ", df.shape[1] - 1)

## How many many different classes exist in the dataset?

Note that first we transform the 'Label' column from Object to String

In [ ]:
display("Number of classes: ", len(df["label"].unique()))
display("Number of examples per class:\n", df["label"].value_counts())

## View missing value information

In [ ]:
display("Total number of missing values", df.isnull().sum().sum())
display("Total number of missing values in each column", df.isnull().sum())
display(
    "percentage of missing values in each column", df.isnull().sum() / df.shape[0] * 100
)

# Which featiures are not numerical?

In [ ]:
display("Non numerical features:\n", df.select_dtypes(exclude=[np.number]).columns)

label encoding the non-numerical features

In [ ]:
from sklearn.preprocessing import LabelEncoder
le_tmp = LabelEncoder()
for column in df.select_dtypes(exclude=[np.number]).columns:
    if column != "label":
        df[column] = le_tmp.fit_transform(df[column].astype(str))
        display(f"Encoded {column} with {len(le_tmp.classes_)} classes: {le_tmp.classes_}")

# Data Cleaning

## First we Clean the data by performing the following steps:

1. Drop empty or single valued columns

In [ ]:
df.dropna(axis=1, how="all", inplace=True)

df = df.loc[:, df.apply(pd.Series.nunique) != 1]

2. Drop empty rows

In [ ]:
df.dropna(axis=0, how="all", inplace=True)

3. Drop duplicate rows (wait)
3. (alternative) add a column to count the number of duplicates and then drop
   duplicates. This way we can keep track of the repeated rows.

In [ ]:
# df[" frequency"] = df.groupby(df.columns.tolist(), sort=False).cumcount()
# df[" frequency"] = df[" frequency"].replace(np.nan, 0)
# tmp_df_len = df.shape[0]
df.drop_duplicates(inplace=True)
# display("Number of removed duplicates: ", tmp_df_len - df.shape[0])
#
# display(df.sort_values(" frequency", ascending=False))

4. Mark the values that are Infinity with -2 and NaN with -1

In [ ]:
df.replace([np.inf, -np.inf], -2, inplace=True)
df.replace(np.nan, -1, inplace=True)

5. Drop columns with more than 50% Zero (0) values

In [ ]:
# calculate the percentage of zeros in each column
zero_percentage = (df == 0).sum() / df.shape[0] * 100
display("Percentage of zeros in each column", zero_percentage)
# zero_percentage = zero_percentage.drop(" frequency")  # we need the frequency column
columns_to_drop = zero_percentage[zero_percentage > 50].index.tolist()
display("Columns to drop", columns_to_drop)
df.drop(columns=columns_to_drop, inplace=True)

6. Drop columns with more than 30% (`-2` + `-1`) values

In [ ]:
# calculate the percentage of -2 and -1 in each column
missing_percentage = (df == -2).sum() / df.shape[0] * 100
missing_percentage += (df == -1).sum() / df.shape[0] * 100
display("Percentage of missing values in each column", missing_percentage)
df = df.loc[:, missing_percentage <= 30]

7. Drop rows with atleast one -1 or -2 values

In [ ]:
# df = df[(df != -1).all(axis=1)]
# df = df[(df != -2).all(axis=1)]

8. Transform the ' Label' column with LabelEncoder

In [ ]:
from sklearn.preprocessing import LabelEncoder

features = df.drop(columns=["label"])
label = df["label"]

le = LabelEncoder()
label = le.fit_transform(label)
display("Unique labels: ", le.classes_)
display("Mapped labels: ", le.transform(le.classes_))

9. Cast all the columns except the ' Label' column to float

In [ ]:
features = features.astype(float)

# Scaling

Data scaling allows the algorithm to converge faster and perform better.
We will use the StandardScaler to scale the data.

Note that we don't want to scale the target column ' Label', since
it is a categorical column.

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
features = pd.DataFrame(scaler.fit_transform(features), columns=features.columns)

df = pd.concat([features, pd.DataFrame(label, columns=["label"])], axis=1)

# Visualization

## correlation matrix

In [ ]:
import klib

klib.corr_plot(df, annot=False)

## correlation to target

In [ ]:
klib.corr_plot(df, target="label")

## Data distribution

In [ ]:
klib.dist_plot(df["label"])

## High correlation filter

Here I can remove some of the data with high correlation, but I'm lazy and
I'm not sure if it's a good aproach for this dataset.

Here I only show high correlation features.

In [ ]:
df_tmp = df.copy()
label_encoder = LabelEncoder()
for column in df_tmp.select_dtypes(include=["object"]).columns:
    df_tmp[column] = label_encoder.fit_transform(df_tmp[column].astype(str))
df_tmp = (df_tmp - df_tmp.mean()) / df_tmp.std()
correlation = df_tmp.corr()

for column in correlation.columns:
    display(correlation[column].sort_values(ascending=False).head(2))

# garbage collection

In [ ]:
del df_tmp
del zero_percentage
del missing_percentage
del scaler
del correlation
del label_encoder
# del tmp_df_len
del path
del all_files
del column
del label
del features
gc.collect()

# classical ML

In [ ]:
from sklearn.model_selection import train_test_split

df_tmp = df.copy()

X = df_tmp.drop("label", axis=1)
y = df_tmp["label"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

## Train the 5 models

ExtraTreesClassifier ranked first
RandomForestClassifier ranked second
BaggingClassifier ranked fourth
GaussianNB ranked seventh
AdaBoostClassifier ranked seventeenth

In [ ]:
from sklearn.ensemble import ExtraTreesClassifier, RandomForestClassifier, BaggingClassifier, AdaBoostClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score, classification_report
from yellowbrick.classifier import ClassificationReport, ConfusionMatrix
import numpy as np
import pandas as pd
import time
import dill


models = {
    "ExtraTreesClassifier": ExtraTreesClassifier(),
    "RandomForestClassifier": RandomForestClassifier(),
    "BaggingClassifier": BaggingClassifier(),
    "GaussianNB": GaussianNB(),
    "AdaBoostClassifier": AdaBoostClassifier(),
}

models_info_dict = {}

# Train models and collect performance metrics
for model_name, model in models.items():
    start_time = time.time()
    model.fit(X_train, y_train)
    end_time = time.time()
    train_time = end_time - start_time

    y_pred = model.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)
    report = classification_report(y_test, y_pred, output_dict=True)

    models_info_dict[model_name] = {
        "accuracy": accuracy,
        "train_time": train_time,
        "classification_report": report,
    }
    # save the model
    with open(f"{model_name}.pkl", "wb") as f:
        dill.dump(model, f)

### plotting

Extract data from models_info_dict

In [ ]:
model_names = list(models_info_dict.keys())
accuracies = [models_info_dict[model]['accuracy'] for model in model_names]
train_times = [models_info_dict[model]['train_time'] for model in model_names]

### 1. Accuracy Comparison Bar Plot

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(10, 6))
sns.barplot(x=accuracies, y=model_names, palette='viridis')
plt.title('Model Accuracy Comparison')
plt.xlabel('Accuracy')
plt.ylabel('Models')
plt.show()
plt.savefig("accuracy_comparison.png")

#### 2. Training Time Comparison Bar Plot

In [ ]:
plt.figure(figsize=(10, 6))
sns.barplot(x=train_times, y=model_names, palette='magma')
plt.title('Model Training Time Comparison')
plt.xlabel('Training Time (seconds)')
plt.ylabel('Models')
plt.show()
plt.savefig("training_time_comparison.png")

#### 3. Precision, Recall, F1-Score Heatmaps

In [ ]:
def plot_classification_report(report, title):
    df_report = pd.DataFrame(report).transpose().drop(['support'], axis=1)
    plt.figure(figsize=(8, 6))
    sns.heatmap(df_report.iloc[:-1, :], annot=True, cmap='coolwarm', fmt='.2f')
    plt.title(title)
    plt.show()
    plt.savefig(f"{title}.png")


for model_name, model_info in models_info_dict.items():
    plot_classification_report(model_info['classification_report'], f'{model_name} Classification Report')

#### 4. Radar Chart

In [ ]:
from math import pi


def plot_radar_chart(models_info_dict):
    categories = ['accuracy', 'precision', 'recall', 'f1-score']
    num_vars = len(categories)

    for model_name, model_info in models_info_dict.items():
        report = model_info['classification_report']['weighted avg']
        values = [model_info['accuracy'], report['precision'], report['recall'], report['f1-score']]
        values += values[:1]  # Repeat the first value to close the circle

        angles = [n / float(num_vars) * 2 * pi for n in range(num_vars)]
        angles += angles[:1]

        plt.figure(figsize=(8, 8))
        ax = plt.subplot(111, polar=True)
        plt.xticks(angles[:-1], categories, color='grey', size=12)
        ax.plot(angles, values, linewidth=2, linestyle='solid', label=model_name)
        ax.fill(angles, values, alpha=0.4)
        plt.title(f'{model_name} Performance Radar Chart', size=16)
        plt.legend(loc='upper right', bbox_to_anchor=(0.1, 0.1))
        plt.show()
        plt.savefig(f"{model_name}_radar_chart.png")

#### 5. Confusion Matrix Visualization

In [ ]:
from sklearn.metrics import confusion_matrix
import itertools


def plot_confusion_matrix(y_true, y_pred, classes, title='Confusion matrix', cmap=plt.cm.Blues):
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(8, 6))
    plt.imshow(cm, interpolation='nearest', cmap=cmap)
    plt.title(title)
    plt.colorbar()
    tick_marks = np.arange(len(classes))
    plt.xticks(tick_marks, classes, rotation=45)
    plt.yticks(tick_marks, classes)

    fmt = 'd'
    thresh = cm.max() / 2.
    for i, j in itertools.product(range(cm.shape[0]), range(cm.shape[1])):
        plt.text(j, i, format(cm[i, j], fmt), horizontalalignment="center",
                 color="white" if cm[i, j] > thresh else "black")

    plt.ylabel('True label')
    plt.xlabel('Predicted label')
    plt.tight_layout()
    plt.savefig(f"{title}.png")


# Plot confusion matrices for each model
for model_name, model in models.items():
    y_pred = model.predict(X_test)
    plot_confusion_matrix(y_test, y_pred, classes=np.unique(y_test), title=f'{model_name} Confusion Matrix')

# Plot radar charts for each model
plot_radar_chart(models_info_dict)

## 5-fold cross validation

NOTE: This step is commented out because it takes a long time to run.

In [ ]:
# from sklearn.model_selection import cross_val_score
#
#
# for model_name, model in models.items():
#     scores = cross_val_score(model, X, y, cv=5)
#     print(f"{model_name}: {scores.mean()}")

# 4 best features from dataset overall

In [ ]:
from sklearn.preprocessing import MinMaxScaler
from sklearn.feature_selection import SelectKBest
from sklearn.feature_selection import chi2

# Scale the data again using the min-max scaler since chi2 requires
# non-negative values
scaler = MinMaxScaler()
X = pd.DataFrame(scaler.fit_transform(X), columns=X.columns)

bestfeatures = SelectKBest(score_func=chi2, k=10)
fit = bestfeatures.fit(X, y)
dfscores = pd.DataFrame(fit.scores_)
dfcolumns = pd.DataFrame(X.columns)

featureScores = pd.concat([dfcolumns, dfscores], axis=1)
featureScores.columns = ["Specs", "Score"]
display(featureScores.nlargest(4, "Score"))

## 4 best features from dataset based on each class

NOTE: This step is commented out because it takes a long time to run.

In [ ]:
# import shap
# from sklearn.ensemble import RandomForestClassifier
#
#
# model = RandomForestClassifier(random_state=42)
# model.fit(X, y)
# explainer = shap.TreeExplainer(model)
# shap_values = explainer.shap_values(X)
#
# for i, class_name in enumerate(model.classes_):
#     print(f"\nTop 4 features for {class_name}:")
#     shap.summary_plot(shap_values[i], X, plot_type="bar", max_display=4)

# Dendritic Cell Algorithm

methodology:
1. PAMP: High, positive correlated features in attack only dataframe
2. Danger: High, positive correlated features in full dataframe
3. Safe: Low, negative correlated features in full dataframe + Low, negative correlated features in attack only dataframe

each will have 4 features, in the Safe case, 2 features from the full dataframe and 2 features from the attack only dataframe

Crolation analysis

In [ ]:
df_tmp = df.copy()
df_tmp.columns = df_tmp.columns.str.strip()

X = df_tmp.drop("label", axis=1)
y = df_tmp["label"]


X_y = X.copy()
X_y["label"] = y
corr_all = X_y.corr()["label"].drop("label")

df_attack = df_tmp[df_tmp["label"] != 0].copy()
corr_attack = df_attack.corr()["label"].drop("label")

In [ ]:
# PAMP: High, positive correlated features in attack only dataframe
pamp_features = corr_attack.nlargest(4).index.tolist()
# Danger: High, positive correlated features in full dataframe
# make sure to not include the same features as in PAMP, 4 top features that are not in PAMP
tmp = [f for f in corr_all.index if f not in pamp_features]
danger_features = corr_all[tmp].nlargest(4).index.tolist()
# Safe: Low, negative correlated features in full dataframe + Low, negative correlated features in attack only dataframe
# make sure to not include the same features as in PAMP and Danger
tmp = [f for f in corr_all.index if f not in pamp_features + danger_features]
safe_features = corr_all[tmp].nsmallest(2).index.tolist() + corr_attack[tmp].nsmallest(2).index.tolist()

In [ ]:
# Map Features to Signals
signals = {
    "PAMP": X[pamp_features].sum(axis=1),
    "Danger": X[danger_features].sum(axis=1),
    "Safe": X[safe_features].sum(axis=1),
}

In [ ]:
# garbage collection
del tmp
del X_y
del corr_all
del corr_attack
del df_attack
gc.collect()

In [ ]:
# Dendritic Cell Algorithm
class DendriticCell:
    def __init__(self, lifespan=100):
        self.lifespan = lifespan
        self.age = 0
        self.k = 0
        self.data_indices = []
        self.labels = []

    def process(self, index, pamp, danger, safe, label):
        self.age += 1
        w_pamp = 2.0
        w_danger = 1.0
        w_safe = 0.5
        self.k += w_pamp * pamp + w_danger * danger - w_safe * safe
        self.data_indices.append(index)
        self.labels.append(label)

    def is_mature(self):
        return self.age >= self.lifespan

In [ ]:
import random
from collections import defaultdict

num_cells = 100
cells = [DendriticCell(random.randint(80, 120)) for _ in range(num_cells)]

context_table = defaultdict(list)
k_values_by_index = defaultdict(list)

start_time = time.time()
for i in range(len(X)):
    pamp = signals["PAMP"].iloc[i]
    danger = signals["Danger"].iloc[i]
    safe = signals["Safe"].iloc[i]
    label_i = y.iloc[i]

    active_cells = random.sample(cells, k=int(num_cells * 0.1))

    for dc in active_cells:
        dc.process(i, pamp, danger, safe, label_i)

        if dc.is_mature():
            if dc.labels:
                majority_label = pd.Series(dc.labels).value_counts().idxmax()
                for idx in dc.data_indices:
                    context_table[idx].append(majority_label)
                    k_values_by_index[idx].append(dc.k)
            dc.__init__(random.randint(80, 120))
end_time = time.time()
dca_time = end_time - start_time

In [ ]:
from sklearn.utils.class_weight import compute_class_weight

class_labels = np.unique(y)
weights_array = compute_class_weight('balanced', classes=class_labels, y=y)
class_weights = dict(zip(class_labels, weights_array))

In [ ]:
y_pred = []
for i in range(len(X)):
    votes = context_table.get(i, [])
    if not votes:
        pred = 0  # fallback class (e.g., BENIGN)
    else:
        vote_series = pd.Series(votes)
        vote_counts = vote_series.value_counts()
        weighted_votes = {cls: class_weights.get(cls, 1.0) * count for cls, count in vote_counts.items()}
        pred = max(weighted_votes.items(), key=lambda x: x[1])[0]
    y_pred.append(pred)

## Evaluation

### reports

In [ ]:
print("Training time: ", dca_time)
dca_accuracy = np.mean(np.array(y_pred) == y.values)
print("Accuracy: ", dca_accuracy)
dca_report = classification_report(y, y_pred, output_dict=True)

### precision recall f1 score heatmap

In [ ]:
plot_classification_report(dca_report, "DCA Classification Report")

### radar chart

In [ ]:
plot_radar_chart({
    "DCA": {
        "accuracy": dca_accuracy,
        "train_time": dca_time,
        "classification_report": dca_report,
    }
})

### confusion matrix

In [ ]:
plot_confusion_matrix(y, y_pred, classes=np.unique(y), title='DCA Confusion Matrix')

## 3. GA optimization of DCA

In [ ]:
df_tmp = df.copy()
df_tmp.columns = df_tmp.columns.str.strip()

X = df_tmp.drop("label", axis=1)
y = df_tmp["label"]

In [ ]:
# Signal correlation
X_y = X.copy()
X_y["label"] = y
corr_all = X_y.corr()["label"].drop("label")
df_attack = df_tmp[df_tmp["label"] != 0].copy()
corr_attack = df_attack.corr()["label"].drop("label")

pamp_features = corr_attack.nlargest(4).index.tolist()
tmp = [f for f in corr_all.index if f not in pamp_features]
danger_features = corr_all[tmp].nlargest(4).index.tolist()
tmp = [f for f in corr_all.index if f not in pamp_features + danger_features]
safe_features = corr_all[tmp].nsmallest(2).index.tolist() + corr_attack[tmp].nsmallest(2).index.tolist()

signals = {
    "PAMP": X[pamp_features].sum(axis=1),
    "Danger": X[danger_features].sum(axis=1),
    "Safe": X[safe_features].sum(axis=1),
}

In [ ]:
def dca_fitness(weight):
    class DCAG:
        def __init__(self, lifespan=100):
            self.lifespan = lifespan
            self.age = 0
            self.k = 0
            self.data_indices = []
            self.labels = []

        def process(self, index, pamp, danger, safe, label):
            self.age += 1
            self.k += weights[0] * pamp + weights[1] * danger - weights[2] * safe
            self.data_indices.append(index)
            self.labels.append(label)

        def is_mature(self):
            return self.age >= self.lifespan

    num_cells = 100
    cells = [DendriticCell(random.randint(80, 120)) for _ in range(num_cells)]
    context_table = defaultdict(list)

    for i in range(len(X)):
        pamp = signals["PAMP"].iloc[i]
        danger = signals["Danger"].iloc[i]
        safe = signals["Safe"].iloc[i]
        label_i = y.iloc[i]

        active_cells = random.sample(cells, k=int(num_cells * 0.1))

        for dc in active_cells:
            dc.process(i, pamp, danger, safe, label_i)
            if dc.is_mature():
                if dc.labels:
                    majority_label = pd.Series(dc.labels).value_counts().idxmax()
                    for idx in dc.data_indices:
                        context_table[idx].append(majority_label)
                dc.__init__(random.randint(80, 120))

    y_pred = []
    for i in range(len(X)):
        votes = context_table.get(i, [])
        if not votes:
            pred = 0
        else:
            vote_series = pd.Series(votes)
            pred = vote_series.value_counts().idxmax()
        y_pred.append(pred)

    report = classification_report(y, y_pred, output_dict=True)
    return report["accuracy"],

In [ ]:
from deap import base, creator, tools, algorithms

creator.create("FitnessMax", base.Fitness, weights=(1.0,))
creator.create("Individual", list, fitness=creator.FitnessMax)

toolbox = base.Toolbox()
toolbox.register("attr_float", random.uniform, 0.1, 5.0)
toolbox.register("individual", tools.initRepeat, creator.Individual, toolbox.attr_float, n=3)
toolbox.register("population", tools.initRepeat, list, toolbox.individual)
toolbox.register("evaluate", dca_fitness)
toolbox.register("mate", tools.cxBlend, alpha=0.5)
toolbox.register("mutate", tools.mutGaussian, mu=1, sigma=0.5, indpb=0.5)
toolbox.register("select", tools.selTournament, tournsize=3)

population = toolbox.population(n=10)
algorithms.eaSimple(population, toolbox, cxpb=0.6, mutpb=0.3, ngen=5, verbose=False)

best_ind = tools.selBest(population, k=1)[0]
best_ind

# Neural Networks

## 1. FastAI Tabular

Note: technically, this needs weight balancing, since the dataset is
hugly imbalanced towards the 'Benign' class. But thet version use a lot of
memory and crashes the kernel.

In [ ]:
from fastai.tabular.all import *

df_tmp = df.copy()

dls = TabularDataLoaders.from_df(
    df_tmp,
    procs=[Categorify, Normalize],
    cont_names=df_tmp.columns.difference(["label"]).tolist(),
    y_names="label",
    valid_pct=0.2,
    seed=42,
)

start_time = time.time()
learn = tabular_learner(dls, metrics=accuracy)
learn.fit_one_cycle(5)
end_time = time.time()
fastai_train_time = end_time - start_time
print(f"Training time: {fastai_train_time:.2f} seconds")

### report

In [ ]:
# Predict on validation set
val_dl = dls.valid
preds, targs = learn.get_preds(dl=val_dl)
y_pred = np.argmax(preds, axis=1)
y_true = targs.numpy()

# Classification Report
print("Classification Report:\n", classification_report(y_true, y_pred))

In [ ]:
# Confusion Matrix
cm = confusion_matrix(y_true, y_pred)
sns.heatmap(cm, annot=True, fmt='d')
plt.title("Confusion Matrix")
plt.show()

In [ ]:
# Precision-Recall Curve
prec, rec, _ = precision_recall_curve(y_true, preds[:,1])
plt.plot(rec, prec)
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision-Recall Curve")
plt.show()

In [ ]:
# Compute average k-values for each index and collect labels
k_avg = []
labels_for_k = []

for idx in k_values_by_index:
    avg_k = np.mean(k_values_by_index[idx])
    k_avg.append(avg_k)
    labels_for_k.append(y.iloc[idx])  # Multi-class label

# Group k-values by label
from collections import defaultdict

k_by_class = defaultdict(list)
for k, label in zip(k_avg, labels_for_k):
    k_by_class[label].append(k)

# Prepare plot
plt.figure(figsize=(12, 6))
bins = 100
for label, k_vals in k_by_class.items():
    plt.hist(k_vals, bins=bins, alpha=0.6, label=str(label))

plt.axvline(0, color='black', linestyle='--')
plt.title("Distribution of k-values by Class")
plt.xlabel("k-value")
plt.ylabel("Number of Samples")
plt.legend(title="Class Label")
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
explainer = shap.Explainer(learn.model.cpu(), torch.from_numpy(df_fastai.iloc[val_dl.items]['Label'].values.astype(np.float32)))
shap_values = explainer(torch.from_numpy(df_fastai.iloc[val_dl.items].drop(columns='Label').values.astype(np.float32)))

shap.plots.beeswarm(shap_values)

## 2. FastAI Tabular with sampling

In [ ]:
from fastai.tabular.all import *

df_tmp = df.copy()

X = df_tmp.drop("label", axis=1)
y = df_tmp["label"]

undersample the 'Benign' class

In [ ]:
from imblearn.under_sampling import RandomUnderSampler

rus = RandomUnderSampler(random_state=42)
X_resampled, y_resampled = rus.fit_resample(X, y)

train the model

In [ ]:
dls = TabularDataLoaders.from_df(
    pd.concat([X_resampled, pd.DataFrame(y_resampled, columns=["label"])]),
    procs=[Categorify, Normalize],
    cont_names=X.columns.tolist(),
    y_names="label",
    valid_pct=0.2,
    seed=42,
)

start_time = time.time()
learn = tabular_learner(dls, metrics=accuracy)
learn.fit_one_cycle(5)
end_time = time.time()
fastai_train_time = end_time - start_time
print(f"Training time: {fastai_train_time:.2f} seconds")

### report

In [ ]:
# Predict on validation set
val_dl = dls.valid
preds, targs = learn.get_preds(dl=val_dl)
# ERROR: Input y_true contains NaN.
y_pred = np.argmax(preds, axis=1)
y_true = targs.numpy()
# Classification Report
print("Classification Report:\n", classification_report(y_true, y_pred))

In [ ]:
# Confusion Matrix
cm = confusion_matrix(y_true, y_pred)
sns.heatmap(cm, annot=True, fmt='d')
plt.title("Confusion Matrix")
plt.show()

In [ ]:
# ROC Curve
fpr, tpr, _ = roc_curve(y_true, preds[:,1])
roc_auc = auc(fpr, tpr)
plt.plot(fpr, tpr, label=f"ROC AUC = {roc_auc:.2f}")
plt.plot([0, 1], [0, 1], "k--")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve")
plt.legend()
plt.show()

In [ ]:
# Precision-Recall Curve
prec, rec, _ = precision_recall_curve(y_true, preds[:,1])
plt.plot(rec, prec)
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision-Recall Curve")
plt.show()

In [ ]:
explainer = shap.Explainer(learn.model.cpu(), torch.from_numpy(df_fastai.iloc[val_dl.items]['Label'].values.astype(np.float32)))
shap_values = explainer(torch.from_numpy(df_fastai.iloc[val_dl.items].drop(columns='Label').values.astype(np.float32)))
shap.plots.beeswarm(shap_values)